# Hafta 03: Uygulama Katmanı (Application Layer)
## HTTP, DNS, Web Cache, Çerezler, SMTP/E-Posta, FTP ve Soketler

> **Ders Kitabı Referansı:** Computer Networking: A Top-Down Approach (J.F. Kurose, K.W. Ross — Bölüm 2)

---

## 1. Temel Kavramlar ve Mimari Çerçeve

Sosyal ağlar, online oyunlar, video konferans, web servisleri ve e-posta sistemleri doğrudan **Uygulama Katmanında (Application Layer)** çalışır. Bu katman, kullanıcının doğrudan temas ettiği veya yazılımcının geliştirdiği süreçlerin yer aldığı yerdir; ancak alt katmanlarla sürekli işbirliği halindedir.

Örneğin bir web tarayıcısına `www.cbu.edu.tr` yazıldığında arka planda şu protokol orkestrasyonu gerçekleşir:
1. **DHCP:** Makinenin yerel ağa bağlanırken kendisine ait IP adresini, varsayılan ağ geçidini ve ISS DNS sunucu IP'sini talep etmesi.
2. **DNS:** İnsanların kolay okuduğu `www.cbu.edu.tr` alan adını yönlendiricilerin işleyebileceği 32-bitlik IP adresine (`194.27.156.248`) çevirmesi.
3. **HTTP / HTTPS:** Hedef sunucudan web nesnelerini (`HTML`, `CSS`, `Görsel`, `JSON`) çekmek için istemci-sunucu diyaloğunu yürütmesi.

---

### 1.1. Ağ Uygulama Mimarileri: Client-Server (CS) vs Peer-to-Peer (P2P)

| Karşılaştırma Kriteri | İstemci-Sunucu (Client-Server) | Eşler Arası (Peer-to-Peer / P2P) |
| :--- | :--- | :--- |
| **Merkeziyetçilik** | Merkezi ve **sabit IP'li** bir sunucu (Server) her zaman açık kalır. | Merkezi sunucu yoktur; cihazlar (**Peers**) doğrudan birbirine bağlanır. |
| **İletişim Akışı** | İstemciler birbirleriyle doğrudan konuşamaz; tüm trafik sunucudan geçer. | Her düğüm hem veri indiren (Client) hem veri yükleyen (Server) konumundadır. |
| **Ölçeklenebilirlik** | Kullanıcı arttıkça sunucu darboğaza girer; devasa **Veri Merkezleri (Data Centers)** gerektirir. | **Kendini Ölçekleme (Self-scalability)** vardır; her yeni eş ağa işlem gücü ve bant genişliği katar. |
| **Örnekler** | Web (HTTP), E-Posta (SMTP), Dosya (FTP) | BitTorrent, Dağıtık Blokzincir Ağları |

---

### 1.2. Uygulama İletişiminin Temeli: Soket, IP ve Port Numaraları

Bir ağ uygulaması diğer bir ağ uygulamasıyla **Soket API (IP + Port)** üzerinden haberleşir.
* **IP Adresi:** İnternetteki hedef fiziksel makineyi bulur (32-bit IPv4 / 128-bit IPv6). Soldan sağa doğru gidildikçe cihazın bağlı olduğu ağa yaklaşılır.
* **Port Numarası:** O makinede aynı anda internete erişen süreçler arasından hedef uygulamayı ayırt eden 16-bitlik kimliktir.

> **SORU: Port numaraları neden 65535 ile sınırlıdır?**  
> **CEVAP:** Taşıma Katmanı protokolleri (TCP ve UDP), paket başlığı tasarımında port alanları için **16 bit** ayırmıştır. $2^{16} = 65536$ farklı port adresi üretilebilir (`0` rezerve olduğundan `1 - 65535` arası kullanılır).
>
> * **0 - 1023 (Well-Known / Tanımlı Portlar):** HTTP (80), HTTPS (443), DNS (53), SMTP (25), FTP Kontrol (21), FTP Veri (20).
> * **1024 - 49151 (Registered Portlar):** Veritabanı ve kurumsal servisler (PostgreSQL: 5432, MySQL: 3306, Redis: 6379).
> * **49152 - 65535 (Dinamik / Ephemeral Portlar):** İstemcilerin geçici bağlantılarında işletim sisteminin anlık dağıttığı çıkış kapılarıdır.

---

### 1.3. Uygulamanın İhtiyaç Duyduğu 4 Temel Servis ve Taşıma Seçimi

Bir ağ uygulaması veri alışverişi için şu dört servise ihtiyaç duyar:
1. **Kayıpsız İletim (Reliability):** Paketlerin router kuyruklarında düşmeden eksiksiz iletilmesi (EFT, e-posta, dosya transferi için şarttır). TCP sağlar, UDP sağlamaz.
2. **Başarım (Throughput):** Hat kapasitesinin verimli kullanılması; iletim hızının dinamik ayarlanması (TCP akış ve tıkanıklık kontrolüyle destekler).
3. **Zamanlama (Timing):** Gecikmenin minimum olması (Oyun ve ses iletiminde <100ms). Donanımsal hat hızına bağlıdır; protokol garantisi yoktur.
4. **Güvenlik (Security):** Verinin şifrelenmesi. Standart TCP/IP katmanları bu hizmeti sunmaz; güvenlik SSL/TLS katmanından alınır.

> **SORU: TCP kayıpsız ve güvenilirken neden bazı uygulamalar özellikle UDP seçer?**  
> **CEVAP:** TCP güvenilirdir ancak yavaştır. 3'lü el sıkışma (*Three-Way Handshake*) kurar, kaybolan paketi yeniden ister, paket sırasını bekler ve CPU maliyeti yüksektir. Gerçek zamanlı sistemlerde (ses, video, oyun) küçük paket kayıpları tolere edilir ancak gecikme tolere edilemez.
>
> **Kritik Mimari Fark (Broadcast):** TCP yalnızca bire bir (Point-to-Point) bağlantı kurar. UDP ise tek bir paketi yerel ağdaki herkese yayabilir (**Broadcast**). Bu sebeple ağa bağlanırken henüz IP'si olmayan ya da hızlı sorgu atan **DNS, DHCP, SNMP ve RIP** protokolleri UDP üzerinde çalışır.

---

### 1.4. Web, HTTP ve Durumsuzluk (Statelessness)

Bir URL 4 bileşenden oluşur: `protocol://hostname:port/path-and-file-name`.
* **Durumsuz Protokol (Stateless):** HTTP sunucusu daha önce bağlanan kullanıcının bilgilerini belleğinde tutmaz; her istek bağımsızdır.
* **Cookie (Çerez):** Durumsuzluğu aşmak için sunucu ilk yanıtta `Set-Cookie: ID=1678` başlığı döner. Tarayıcı bu ID'yi diske kaydeder ve sonraki tüm ziyaretlerinde sunucuya ileterek kendini tanıtır (Sepet, oturum, kişiselleştirilmiş reklam).
* **Web Cache (Proxy):** İstemcinin doğrudan orijinal sunucuya gitmesi yerine en yakındaki önbellek sunucusundan sayfayı almasıdır. İnternet çıkış trafiğini azaltır ve erişimi hızlandırır.
* **Conditional GET:** Önbellekteki nesnenin güncelliğini denetlemek için HTTP başlığında `If-Modified-Since:` satırı kullanılır. Dosya değişmemişse sunucu veri göndermez; sadece **`304 Not Modified`** döner (Sıfır gövde aktarımıyla hat tasarrufu sağlanır).

#### Standart HTTP Durum Kodları (Status Codes)
| Kod | Anlamı | Açıklama |
| :--- | :--- | :--- |
| **200 OK** | Başarılı | İstenen nesne bulundu ve yanıt gövdesinde teslim edildi. |
| **301 Moved Permanently** | Kalıcı Yönlendirme | İstenen URL kalıcı olarak yeni bir adrese taşındı. |
| **304 Not Modified** | Değiştirilmedi | Conditional GET sorgusunda nesnenin değişmediğini bildirir. |
| **400 Bad Request** | Hatalı İstek | Sunucunun anlayamadığı sözdizimi hatası içeren istek. |
| **404 Not Found** | Bulunamadı | İstenen belge sunucuda mevcut değil. |
| **505 HTTP Version Not Supported** | Sürüm Desteklenmiyor | Sunucu istemcinin HTTP protokol sürümünü desteklemiyor. |

---

### 1.5. DNS Hiyerarşisi ve Dağıtık Veritabanı Mantığı

İnternet cihazları insan dostu alan adları (`www.cbu.edu.tr`) yerine yönlendiricilerin işleyebileceği 32-bit IP adreslerini kullanır. DNS (Domain Name System), bu dönüşümü yapan dağıtık bir veritabanıdır.
* **Forward Mapping:** Alan adından IP adresine dönüşüm (`www.cbu.edu.tr -> 194.27.156.248`).
* **Reverse Mapping (PTR):** IP adresinden resmi sunucu adına dönüşüm (`8.8.8.8 -> dns.google`).
* **Neden Tek Bir Merkezi DNS Sunucusu Yoktur?** Tek bir merkez olsaydı, o sunucunun çökmesi tüm dünya internetini durdururdu (*Single Point of Failure*), milyarlarca sorgu tek noktada tıkanıklık yaratır ve uzak kıtalarda devasa sorgu gecikmeleri oluşurdu. Bu sebeple DNS; **Kök (Root)**, **Üst Düzey Alan Adı (TLD: .tr, .com, .edu)** ve **Yetkili (Authoritative)** sunuculardan oluşan hiyerarşik bir ağaç yapısında dağıtılmıştır.

---

### 1.6. E-Posta Mimarisi: SMTP (Push) vs POP3/IMAP (Pull)

E-posta altyapısı üç temel bileşenden oluşur: Kullanıcı Ajanı (User Agent / Outlook / Webmail), Posta Sunucusu (Mail Server) ve İletim Protokolleri.
* **SMTP (Simple Mail Transfer Protocol - Port 25):** Yalnızca **İtme (Push)** protokolüdür. Gönderici istemci postayı kendi sunucusuna iletirken ve sunucular postayı birbirine aktarırken SMTP kullanır.
* **E-Posta Çekme Protokolleri (POP3: Port 110, IMAP: Port 143, HTTP):** SMTP alıcının bilgisayarına doğrudan e-posta atamaz; çünkü alıcının bilgisayarı her an açık değildir. Posta, alıcının sunucusundaki posta kutusunda (Mailbox) bekletilir. Alıcı bilgisayarını açtığında postayı sunucudan **Çekmek (Pull)** için POP3, IMAP veya HTTP protokollerini kullanır.

---
## 2. Mimari Görselleştirmeler (Vektörel SVG)

Aşağıdaki kod hücreleri; Client-Server vs P2P, Hiyerarşik Dağıtık DNS Mimarisi, Web Cache / Conditional GET ve SMTP E-Posta Akışını harici grafik kütüphanesi veya DLL kullanmadan saf SVG ile ekrana basar.

In [ ]:
from IPython.display import SVG, display

# Şekil 1: Client-Server vs Peer-to-Peer (P2P) Mimarisi
svg_cs_p2p = """
<svg width="820" height="280" viewBox="0 0 820 280" xmlns="http://www.w3.org/2000/svg" style="background:#ffffff; font-family:sans-serif;">
  <text x="410" y="24" text-anchor="middle" font-size="14" font-weight="bold" fill="#111827">Ağ Uygulama Mimarileri: İstemci-Sunucu (Client-Server) vs Eşler Arası (Peer-to-Peer)</text>
  
  <!-- SOL: CLIENT-SERVER -->
  <rect x="30" y="40" width="360" height="225" rx="8" fill="#f8f9fa" stroke="#dee2e6" stroke-width="1.5"/>
  <text x="210" y="62" text-anchor="middle" font-size="12" font-weight="bold" fill="#1e3a8a">İstemci - Sunucu (Client-Server)</text>
  <rect x="160" y="80" width="100" height="45" rx="6" fill="#fee2e2" stroke="#dc2626" stroke-width="2"/>
  <text x="210" y="100" text-anchor="middle" font-size="11" font-weight="bold" fill="#991b1b">Merkezi Sunucu</text>
  <text x="210" y="116" text-anchor="middle" font-size="9" fill="#7f1d1d">Sabit IP: 93.184.216.34</text>
  <circle cx="80" cy="200" r="24" fill="#dbeafe" stroke="#2563eb" stroke-width="1.5"/>
  <text x="80" y="204" text-anchor="middle" font-size="10" font-weight="bold" fill="#1e40af">Client 1</text>
  <circle cx="210" cy="200" r="24" fill="#dbeafe" stroke="#2563eb" stroke-width="1.5"/>
  <text x="210" y="204" text-anchor="middle" font-size="10" font-weight="bold" fill="#1e40af">Client 2</text>
  <circle cx="340" cy="200" r="24" fill="#dbeafe" stroke="#2563eb" stroke-width="1.5"/>
  <text x="340" y="204" text-anchor="middle" font-size="10" font-weight="bold" fill="#1e40af">Client 3</text>
  <line x1="95" y1="180" x2="175" y2="125" stroke="#64748b" stroke-width="1.5" stroke-dasharray="3,3"/>
  <line x1="210" y1="176" x2="210" y2="125" stroke="#64748b" stroke-width="1.5" stroke-dasharray="3,3"/>
  <line x1="325" y1="180" x2="245" y2="125" stroke="#64748b" stroke-width="1.5" stroke-dasharray="3,3"/>
  <text x="210" y="250" text-anchor="middle" font-size="10" fill="#475569">İstemciler birbirini görmez; tüm yük sunucuya biner.</text>

  <!-- SAĞ: P2P -->
  <rect x="430" y="40" width="360" height="225" rx="8" fill="#f8f9fa" stroke="#dee2e6" stroke-width="1.5"/>
  <text x="610" y="62" text-anchor="middle" font-size="12" font-weight="bold" fill="#166534">Eşler Arası (Peer-to-Peer / P2P)</text>
  <circle cx="510" cy="110" r="25" fill="#dcfce7" stroke="#16a34a" stroke-width="2"/>
  <text x="510" y="114" text-anchor="middle" font-size="10" font-weight="bold" fill="#14532d">Peer A</text>
  <circle cx="710" cy="110" r="25" fill="#dcfce7" stroke="#16a34a" stroke-width="2"/>
  <text x="710" y="114" text-anchor="middle" font-size="10" font-weight="bold" fill="#14532d">Peer B</text>
  <circle cx="550" cy="200" r="25" fill="#dcfce7" stroke="#16a34a" stroke-width="2"/>
  <text x="550" y="204" text-anchor="middle" font-size="10" font-weight="bold" fill="#14532d">Peer C</text>
  <circle cx="670" cy="200" r="25" fill="#dcfce7" stroke="#16a34a" stroke-width="2"/>
  <text x="670" y="204" text-anchor="middle" font-size="10" font-weight="bold" fill="#14532d">Peer D</text>
  <line x1="535" y1="110" x2="685" y2="110" stroke="#15803d" stroke-width="1.5"/>
  <line x1="510" y1="135" x2="550" y2="175" stroke="#15803d" stroke-width="1.5"/>
  <line x1="710" y1="135" x2="670" y2="175" stroke="#15803d" stroke-width="1.5"/>
  <line x1="575" y1="200" x2="645" y2="200" stroke="#15803d" stroke-width="1.5"/>
  <text x="610" y="250" text-anchor="middle" font-size="10" fill="#166534">Merkezi sunucu yoktur; her düğüm hem Client hem Server'dır.</text>
</svg>
"""
display(SVG(svg_cs_p2p))

In [ ]:
# Şekil 2: Hiyerarşik ve Dağıtık DNS Veritabanı Mimarisi
svg_dns_hierarchy = """
<svg width="820" height="300" viewBox="0 0 820 300" xmlns="http://www.w3.org/2000/svg" style="background:#ffffff; font-family:sans-serif;">
  <text x="410" y="24" text-anchor="middle" font-size="14" font-weight="bold" fill="#111827">Hiyerarşik ve Dağıtık DNS Veritabanı Ağacı</text>
  
  <!-- Root DNS -->
  <g transform="translate(350, 45)">
    <rect width="120" height="36" rx="6" fill="#fee2e2" stroke="#dc2626" stroke-width="2"/>
    <text x="60" y="22" text-anchor="middle" font-size="11" font-weight="bold" fill="#991b1b">Kök (Root) DNS</text>
  </g>
  
  <!-- TLD DNS Seviyesi -->
  <line x1="370" y1="81" x2="170" y2="125" stroke="#6b7280" stroke-width="1.5"/>
  <line x1="410" y1="81" x2="410" y2="125" stroke="#6b7280" stroke-width="1.5"/>
  <line x1="450" y1="81" x2="650" y2="125" stroke="#6b7280" stroke-width="1.5"/>
  
  <g transform="translate(110, 125)">
    <rect width="120" height="36" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
    <text x="60" y="22" text-anchor="middle" font-size="11" font-weight="bold" fill="#92400e">.com TLD Sunucu</text>
  </g>
  <g transform="translate(350, 125)">
    <rect width="120" height="36" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
    <text x="60" y="22" text-anchor="middle" font-size="11" font-weight="bold" fill="#92400e">.tr / .edu TLD Sunucu</text>
  </g>
  <g transform="translate(590, 125)">
    <rect width="120" height="36" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
    <text x="60" y="22" text-anchor="middle" font-size="11" font-weight="bold" fill="#92400e">.org TLD Sunucu</text>
  </g>
  
  <!-- Yetkili (Authoritative) DNS Seviyesi -->
  <line x1="170" y1="161" x2="170" y2="205" stroke="#6b7280" stroke-width="1.5"/>
  <line x1="410" y1="161" x2="410" y2="205" stroke="#6b7280" stroke-width="1.5"/>
  <line x1="650" y1="161" x2="650" y2="205" stroke="#6b7280" stroke-width="1.5"/>
  
  <g transform="translate(95, 205)">
    <rect width="150" height="40" rx="6" fill="#dbeafe" stroke="#2563eb" stroke-width="1.5"/>
    <text x="75" y="20" text-anchor="middle" font-size="10" font-weight="bold" fill="#1e40af">google.com DNS</text>
    <text x="75" y="33" text-anchor="middle" font-size="9" fill="#1e40af">Yetkili Sunucu</text>
  </g>
  <g transform="translate(335, 205)">
    <rect width="150" height="40" rx="6" fill="#dbeafe" stroke="#2563eb" stroke-width="1.5"/>
    <text x="75" y="20" text-anchor="middle" font-size="10" font-weight="bold" fill="#1e40af">cbu.edu.tr DNS</text>
    <text x="75" y="33" text-anchor="middle" font-size="9" fill="#1e40af">194.27.156.248</text>
  </g>
  <g transform="translate(575, 205)">
    <rect width="150" height="40" rx="6" fill="#dbeafe" stroke="#2563eb" stroke-width="1.5"/>
    <text x="75" y="20" text-anchor="middle" font-size="10" font-weight="bold" fill="#1e40af">wikipedia.org DNS</text>
    <text x="75" y="33" text-anchor="middle" font-size="9" fill="#1e40af">Yetkili Sunucu</text>
  </g>
  
  <text x="410" y="280" text-anchor="middle" font-size="10" fill="#4b5563" font-style="italic">
    İstemci önce ISS Yerel DNS'ine sorar; adresi bulana kadar Root -&gt; TLD -&gt; Yetkili sırasıyla sorgu aşağı iner.
  </text>
</svg>
"""
display(SVG(svg_dns_hierarchy))

In [ ]:
# Şekil 3: Web Cache (Proxy) ve Conditional GET (304 Not Modified) Akışı
svg_cache_flow = """
<svg width="820" height="320" viewBox="0 0 820 320" xmlns="http://www.w3.org/2000/svg" style="background:#ffffff; font-family:sans-serif;">
  <defs>
    <marker id="arrow" markerWidth="7" markerHeight="7" refX="5" refY="3.5" orient="auto">
      <polygon points="0 0, 7 3.5, 0 7" fill="#2563eb"/>
    </marker>
    <marker id="arrow-orange" markerWidth="7" markerHeight="7" refX="5" refY="3.5" orient="auto">
      <polygon points="0 0, 7 3.5, 0 7" fill="#d97706"/>
    </marker>
  </defs>

  <text x="410" y="25" text-anchor="middle" font-size="15" font-weight="bold" fill="#111827">
    Web Cache (Proxy) ve Şartlı GET (Conditional GET - 304 Not Modified) Mimarisi
  </text>

  <!-- Cihaz Sütunları -->
  <rect x="60" y="50" width="120" height="40" rx="6" fill="#dbeafe" stroke="#1d4ed8" stroke-width="1.5"/>
  <text x="120" y="74" text-anchor="middle" font-size="11" font-weight="bold" fill="#1e40af">Web Tarayıcı (Host)</text>
  <line x1="120" y1="90" x2="120" y2="300" stroke="#94a3b8" stroke-width="1.5" stroke-dasharray="4,4"/>

  <rect x="350" y="50" width="120" height="40" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
  <text x="410" y="74" text-anchor="middle" font-size="11" font-weight="bold" fill="#92400e">Web Proxy (Cache)</text>
  <line x1="410" y1="90" x2="410" y2="300" stroke="#94a3b8" stroke-width="1.5" stroke-dasharray="4,4"/>

  <rect x="640" y="50" width="130" height="40" rx="6" fill="#fee2e2" stroke="#dc2626" stroke-width="1.5"/>
  <text x="705" y="74" text-anchor="middle" font-size="11" font-weight="bold" fill="#991b1b">Orijinal Web Sunucu</text>
  <line x1="705" y1="90" x2="705" y2="300" stroke="#94a3b8" stroke-width="1.5" stroke-dasharray="4,4"/>

  <!-- 1. Aşama: İlk İstek (Cache Miss) -->
  <line x1="120" y1="120" x2="405" y2="120" stroke="#2563eb" stroke-width="1.5" marker-end="url(#arrow)"/>
  <text x="260" y="114" text-anchor="middle" font-size="10" fill="#1e40af">1. GET /resim.png</text>

  <line x1="410" y1="140" x2="700" y2="140" stroke="#2563eb" stroke-width="1.5" marker-end="url(#arrow)"/>
  <text x="555" y="134" text-anchor="middle" font-size="10" fill="#1e40af">2. Cache Miss -&gt; Sunucuya İlet</text>

  <line x1="705" y1="170" x2="415" y2="170" stroke="#16a34a" stroke-width="1.5" marker-end="url(#arrow)"/>
  <text x="555" y="164" text-anchor="middle" font-size="10" fill="#15803d">3. 200 OK + [Dosya Verisi] + Last-Modified</text>

  <line x1="410" y1="190" x2="125" y2="190" stroke="#16a34a" stroke-width="1.5" marker-end="url(#arrow)"/>
  <text x="260" y="184" text-anchor="middle" font-size="10" fill="#15803d">4. İstemciye İlet ve Önbelleğe Kopyala</text>

  <!-- 2. Aşama: Tekrar İstek (Conditional GET & 304 Not Modified) -->
  <line x1="120" y1="230" x2="405" y2="230" stroke="#d97706" stroke-width="1.5" marker-end="url(#arrow-orange)"/>
  <text x="260" y="224" text-anchor="middle" font-size="10" fill="#b45309">5. Tekrar GET /resim.png</text>

  <line x1="410" y1="250" x2="700" y2="250" stroke="#d97706" stroke-width="1.5" marker-end="url(#arrow-orange)"/>
  <text x="555" y="244" text-anchor="middle" font-size="10" fill="#b45309">6. Conditional GET (If-Modified-Since: Tarih)</text>

  <line x1="705" y1="275" x2="415" y2="275" stroke="#7c3aed" stroke-width="1.5" marker-end="url(#arrow)"/>
  <text x="555" y="269" text-anchor="middle" font-size="10" fill="#6d28d9">7. 304 Not Modified (Veri aktarılmaz, 0 bayt payload!)</text>

  <line x1="410" y1="295" x2="125" y2="295" stroke="#7c3aed" stroke-width="1.5" marker-end="url(#arrow)"/>
  <text x="260" y="289" text-anchor="middle" font-size="10" fill="#6d28d9">8. Önbellekteki kopyayı hızlıca istemciye ver</text>
</svg>
"""
display(SVG(svg_cache_flow))

In [ ]:
# Şekil 4: E-Posta Mimarisi: SMTP (Push) vs POP3/IMAP/HTTP (Pull)
svg_email = """
<svg width="820" height="240" viewBox="0 0 820 240" xmlns="http://www.w3.org/2000/svg" style="background:#ffffff; font-family:sans-serif;">
  <defs>
    <marker id="arr-email" markerWidth="7" markerHeight="7" refX="5" refY="3.5" orient="auto">
      <polygon points="0 0, 7 3.5, 0 7" fill="#2563eb"/>
    </marker>
  </defs>
  <text x="410" y="24" text-anchor="middle" font-size="14" font-weight="bold" fill="#111827">E-Posta Mimarisi: Gönderim (SMTP Push) ve Çekme (POP3/IMAP Pull)</text>
  
  <!-- Alice -->
  <circle cx="80" cy="120" r="30" fill="#dcfce7" stroke="#16a34a" stroke-width="2"/>
  <text x="80" y="118" text-anchor="middle" font-size="11" font-weight="bold" fill="#14532d">Alice</text>
  <text x="80" y="132" text-anchor="middle" font-size="9" fill="#14532d">İstemci</text>
  
  <!-- Alice Mail Sunucusu -->
  <rect x="200" y="90" width="130" height="60" rx="6" fill="#fee2e2" stroke="#dc2626" stroke-width="1.5"/>
  <text x="265" y="118" text-anchor="middle" font-size="11" font-weight="bold" fill="#991b1b">Alice Sunucusu</text>
  <text x="265" y="134" text-anchor="middle" font-size="9" fill="#7f1d1d">mail.alice.com</text>
  
  <!-- Bob Mail Sunucusu -->
  <rect x="470" y="90" width="130" height="60" rx="6" fill="#fee2e2" stroke="#dc2626" stroke-width="1.5"/>
  <text x="535" y="118" text-anchor="middle" font-size="11" font-weight="bold" fill="#991b1b">Bob Sunucusu</text>
  <text x="535" y="134" text-anchor="middle" font-size="9" fill="#7f1d1d">mail.bob.com</text>
  
  <!-- Bob -->
  <circle cx="730" cy="120" r="30" fill="#dbeafe" stroke="#2563eb" stroke-width="2"/>
  <text x="730" y="118" text-anchor="middle" font-size="11" font-weight="bold" fill="#1e40af">Bob</text>
  <text x="730" y="132" text-anchor="middle" font-size="9" fill="#1e40af">İstemci</text>
  
  <!-- Oklar -->
  <line x1="112" y1="120" x2="192" y2="120" stroke="#2563eb" stroke-width="2" marker-end="url(#arr-email)"/>
  <text x="152" y="110" text-anchor="middle" font-size="9" font-weight="bold" fill="#1d4ed8">1. SMTP (Push)</text>
  
  <line x1="332" y1="120" x2="462" y2="120" stroke="#2563eb" stroke-width="2" marker-end="url(#arr-email)"/>
  <text x="397" y="110" text-anchor="middle" font-size="9" font-weight="bold" fill="#1d4ed8">2. SMTP (Push)</text>
  
  <line x1="602" y1="120" x2="692" y2="120" stroke="#16a34a" stroke-width="2" marker-end="url(#arr-email)"/>
  <text x="647" y="110" text-anchor="middle" font-size="9" font-weight="bold" fill="#15803d">3. POP3 / IMAP (Pull)</text>
  
  <text x="410" y="205" text-anchor="middle" font-size="10" fill="#374151">
    SMTP yalnızca gönderme (Push) yapar. Alıcı Bob, sunucusunda bekleyen postayı çekmek (Pull) için POP3/IMAP/HTTP kullanır.
  </text>
</svg>
"""
display(SVG(svg_email))

---
## 3. Deney 1: Saf TCP Soketi ile HTTP GET ve POST (Entity-Body) Ayrıştırması

### 3.1. Deneyin Amacı
Ders notlarınızda belirtilen en önemli sözdizimi farkını kod seviyesinde doğrulamak:
* **GET İsteği:** Belge/nesne talep eder; **Entity-Body (Gövde) boştur**.
* **POST İsteği:** Sunucuya arama formu, JSON veya kullanıcı verisi iletir; **Entity-Body veriyle doludur** ve başlıkta `Content-Length` zorunludur.

In [ ]:
# ==============================================================================
# DENEY 1: Saf Soket ile HTTP GET ve POST (Entity-Body) İnşası
# ==============================================================================
import socket
import json

HEDEF_HOST = "httpbin.org"
HEDEF_PORT = 80

def soket_http_istegi_gonder(istek_metni: str):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(4.0)
    s.connect((HEDEF_HOST, HEDEF_PORT))
    s.sendall(istek_metni.encode('utf-8'))
    
    yanit_bayt = b""
    while True:
        parca = s.recv(1024)
        if not parca:
            break
        yanit_bayt += parca
    s.close()
    return yanit_bayt

# 1. Senaryo: HTTP GET (Entity-Body YOKTUR)
get_istek = (
    "GET /get HTTP/1.1\r\n" +
    f"Host: {HEDEF_HOST}\r\n" +
    "Connection: close\r\n" +
    "\r\n"  # Çift CRLF: Gövde olmadığını gösterir!
)
get_yanit = soket_http_istegi_gonder(get_istek)
get_header, _ = get_yanit.split(b"\r\n\r\n", 1)
print("--- 1. HTTP GET İSTEĞİ (Body Boş) ---")
print(f"Durum Satırı: {get_header.decode('iso-8859-1').splitlines()[0]}")
print(f"Gönderilen Başlık Boyutu: {len(get_istek)} bayt | Entity-Body: 0 bayt\n")

# 2. Senaryo: HTTP POST (Entity-Body DOLUDUR)
post_govde = json.dumps({"ogrenci": "Akin", "bolum": "Yazilim Muh", "lab": "Hafta 3"})
post_istek = (
    "POST /post HTTP/1.1\r\n" +
    f"Host: {HEDEF_HOST}\r\n" +
    "Content-Type: application/json\r\n" +
    f"Content-Length: {len(post_govde)}\r\n" +
    "Connection: close\r\n" +
    "\r\n" +  # Çift CRLF: Başlıklar bitti, gövde başlıyor!
    post_govde
)
post_yanit = soket_http_istegi_gonder(post_istek)
post_header, post_body = post_yanit.split(b"\r\n\r\n", 1)
print("--- 2. HTTP POST İSTEĞİ (Entity-Body Dolu) ---")
print(f"Durum Satırı      : {post_header.decode('iso-8859-1').splitlines()[0]}")
print(f"Gönderilen Gövde  : {post_govde} ({len(post_govde)} bayt)")
print(f"Sunucu Yanıt Özeti: Gönderilen JSON sunucu tarafından başarıyla çözümlendi.")

---
### 3.2. Deney 1'in Mühendislik Yorumu
1. **CRLF (`\r\n`) Sınır Belirteci:** HTTP protokolü metin tabanlıdır. Başlıkların bittiğini ve gövdenin başladığını işletim sistemine bildiren tek işaret çift satır sonu (`\r\n\r\n`) dizisidir.
2. **Entity-Body ve `Content-Length` Zorunluluğu:** GET isteğinde gövde boyutu 0 bayttır. POST isteğinde ise sunucunun soketten kaç bayt okuyacağını bilmesi için `Content-Length` başlığı zorunludur; eksik veya hatalı verilirse soket tamponu tıkanır ve bağlantı zaman aşımına uğrar.

---
## 4. Deney 2: Durumsuz HTTP Sunucu ve Çerez (Cookie) / Oturum Yönetimi Simülasyonu

### 4.1. Deneyin Amacı
HTTP'nin durumsuz (stateless) doğasını bizzat gözlemlemek; bir sunucunun yeni gelen anonim kullanıcıya nasıl `Set-Cookie` kimliği atadığını ve istemcinin sonraki isteklerde bu kimliği göndererek durumunu (oturumunu) nasıl koruduğunu kodlamaktır.

In [ ]:
# ==============================================================================
# DENEY 2: HTTP Çerez (Cookie) ve Oturum Durumu Simülatörü
# ==============================================================================
import random

SUNUCU_OTURUM_TABLOSU = {}

def http_sunucu_isteği_isle(istek_basliklari: dict):
    gelen_cookie = istek_basliklari.get("Cookie")
    
    # Senaryo 1: İlk Kez Giren Anonim Kullanıcı
    if not gelen_cookie or gelen_cookie not in SUNUCU_OTURUM_TABLOSU:
        yeni_oturum_id = f"UID_{random.randint(1000, 9999)}"
        SUNUCU_OTURUM_TABLOSU[yeni_oturum_id] = {"ziyaret_sayisi": 1, "sepet": []}
        return {
            "Status": "200 OK",
            "Set-Cookie": yeni_oturum_id,
            "Body": f"Hoşgeldiniz Anonim Kullanıcı! Size atanan çerez kimliği: {yeni_oturum_id}"
        }
    # Senaryo 2: Tanınan Kullanıcı (Cookie ile gelen)
    else:
        oturum = SUNUCU_OTURUM_TABLOSU[gelen_cookie]
        oturum["ziyaret_sayisi"] += 1
        return {
            "Status": "200 OK",
            "Set-Cookie": None,
            "Body": f"Tekrar hoşgeldiniz {gelen_cookie}! Bu sitedeki {oturum['ziyaret_sayisi']}. ziyaretiniz. Size özel teklifler listelendi."
        }

tarayici_cerez_deposu = {}

print("--- 1. İSTEK: İstemci İlk Kez Amazon'a Giriyor (Çerez Yok) ---")
istek_1 = {"Method": "GET", "Path": "/", "Cookie": tarayici_cerez_deposu.get("amazon_session")}
yanit_1 = http_sunucu_isteği_isle(istek_1)
print(f"Sunucu Yanıtı     : {yanit_1['Status']}")
print(f"Set-Cookie Başlığı: {yanit_1['Set-Cookie']}")
print(f"Sayfa İçeriği     : {yanit_1['Body']}")
tarayici_cerez_deposu["amazon_session"] = yanit_1["Set-Cookie"]

print("\n--- 2. İSTEK: İstemci 5 Dakika Sonra Aynı Sayfayı Yeniliyor ---")
istek_2 = {"Method": "GET", "Path": "/", "Cookie": tarayici_cerez_deposu.get("amazon_session")}
yanit_2 = http_sunucu_isteği_isle(istek_2)
print(f"İstemcinin Yolladığı Cookie : {istek_2['Cookie']}")
print(f"Sayfa İçeriği               : {yanit_2['Body']}")

---
### 4.2. Deney 2'nin Mühendislik Yorumu
1. **Durumsuz Sunucu Verimliliği:** HTTP protokolü durum tutmadığı için sunucu milyonlarca aktif TCP bağlantısının belleğini taşımak zorunda kalmaz. İş biter bitmez soket kapatılır.
2. **Durumun İstemcide Saklanması:** Durum bilgisi istemciye emanet edilir (`Set-Cookie`). Kullanıcının sepeti veya oturum anahtarı sonraki isteklerde `Cookie:` başlığı ile geri gelir; böylece sunucu durumsuz mimarisini bozmadan kişiselleştirilmiş hizmet sunar.

---
## 5. Deney 3: Web Proxy Cache ve Şartlı GET (Conditional GET - 304) Simülatörü

### 5.1. Deneyin Amacı
Proxy sunucularının bant genişliğini nasıl tasarruf ettiğini; `If-Modified-Since` başlığı ile sunucuya sorulan nesnenin değişmediği durumda dönen **`304 Not Modified`** mekanizmasını test etmektir.

In [ ]:
# ==============================================================================
# DENEY 3: Web Proxy Cache & Conditional GET Simülasyonu
# ==============================================================================
ORIGIN_SERVER = {
    "/logo.png": {
        "data": b"[BINER_RESIM_VERISI_100_KB]",
        "last_modified": "Tue, 15 Sep 2026 10:00:00 GMT"
    }
}
PROXY_CACHE = {}

def proxy_request(url_path: str):
    print(f"\n[İSTEMCİ -> PROXY] '{url_path}' için istek gönderildi.")
    if url_path not in PROXY_CACHE:
        print("  [!] CACHE MISS: Nesne yerelde yok. Orijinal sunucudan indiriliyor...")
        sunucu_nesnesi = ORIGIN_SERVER[url_path]
        PROXY_CACHE[url_path] = {
            "cached_data": sunucu_nesnesi["data"],
            "cached_time": sunucu_nesnesi["last_modified"]
        }
        return f"200 OK (Full Download: {len(sunucu_nesnesi['data'])} bayt hatta aktarıldı)"
    
    print("  [*] CACHE HIT: Nesne yerelde var. Orijinal sunucuya 'Conditional GET' soruluyor...")
    cached_time = PROXY_CACHE[url_path]["cached_time"]
    if ORIGIN_SERVER[url_path]["last_modified"] == cached_time:
        print("  [✓] SUNUCU YANITI: '304 Not Modified'. Dosya değişmemiş, ağa veri basılmadı!")
        return "304 Not Modified (Sıfır veri aktarımı, yerel kopya teslim edildi)"
    else:
        print("  [!] SUNUCU YANITI: '200 OK'. Dosya güncellenmiş, yeni kopya indiriliyor.")
        PROXY_CACHE[url_path]["cached_data"] = ORIGIN_SERVER[url_path]["data"]
        PROXY_CACHE[url_path]["cached_time"] = ORIGIN_SERVER[url_path]["last_modified"]
        return "200 OK (Yeni sürüm indirildi)"

print(proxy_request("/logo.png"))
print(proxy_request("/logo.png"))

---
### 5.2. Deney 3'ün Mühendislik Yorumu
1. **Bant Genişliği ve CDN Tasarrufu:** İkinci istekte dönen `304 Not Modified` yanıtı 0 bayt gövde içerir. Bu durum devasa kurumsal ağlarda ve Cloudflare/Akamai gibi CDN yapılarında omurga hat trafiğini %70'e varan oranlarda azaltır.
2. **Önbellek Doğrulama (Cache Invalidation):** Tarayıcı veya proxy dosyanın yerel kopyasını saklarken orijinal sunucudaki güncelliği doğrulamak için `If-Modified-Since` başlığıyla şartlı sorgu atar; böylece bayat veri sunma riski ortadan kalkar.

---
## 6. Deney 4: DNS (UDP Port 53) İstemcisi, Forward/Reverse Mapping ve İşletim Sistemi Keşfi

### 6.1. Deneyin Amacı
DNS servisinin neden **UDP Port 53** üzerinde çalıştığını doğrulamak; işletim sistemi DNS istemcisini kullanarak `Forward Mapping` (Alan Adı -> IP) ve `Reverse Mapping` (IP -> Alan Adı) çözümlemelerini Python üzerinden tetiklemektir.

In [ ]:
# ==============================================================================
# DENEY 4: DNS Çözümleme Mekanizması (Forward ve Reverse Mapping)
# ==============================================================================
import socket

def test_dns_resolution(target_domain: str, target_ip: str):
    print("=" * 65)
    print("DNS (DOMAIN NAME SYSTEM) PROTOKOL ÇÖZÜMLEME ANALİZİ")
    print("=" * 65)
    
    # 1. Forward Mapping (Alan Adı -> IP Numarası)
    print(f"[1] FORWARD MAPPING: '{target_domain}' IP adresi sorgulanıyor...")
    try:
        cozulmus_ip = socket.gethostbyname(target_domain)
        print(f"    -> Çözülen 32-bit IPv4: {cozulmus_ip}")
    except socket.gaierror as e:
        print(f"    -> HATA: Domain çözülemedi: {e}")

    # 2. Reverse Mapping (IP Numarası -> Hostname)
    print(f"\n[2] REVERSE MAPPING: '{target_ip}' hangi sunucuya ait?")
    try:
        hostname, alias_list, ip_list = socket.gethostbyaddr(target_ip)
        print(f"    -> Resmi Hostname: {hostname}")
        print(f"    -> Takma Adlar   : {alias_list}")
    except socket.herror as e:
        print(f"    -> Ters kayıt (PTR) bulunamadı: {e}")

    # 3. Port Bilgisi Doğrulaması
    dns_port = socket.getservbyname("domain", "udp")
    print(f"\n[3] Taşıma Katmanı Eşleşmesi: DNS Servisi = UDP Port {dns_port}")
    print("=" * 65)

test_dns_resolution("www.cbu.edu.tr", "8.8.8.8")

---
### 6.2. Deney 4'ün Mühendislik Yorumu
1. **DNS Neden UDP Kullanır?** Web sayfasına her tıklamada en az bir DNS sorgusu atılır. TCP seçilseydi her sorgu öncesinde 3'lü el sıkışma (Handshake) yapılacak ve bağlantı kurma gecikmesi kullanıcıya yansıyacaktı. UDP ile tek bir istek paketi fırlatılır ve anında yanıt alınır.
2. **Reverse Mapping (Ters Eşleşme - PTR Kayıtları):** IP adresinden alan adını tespit etme işlemi, özellikle e-posta sunucularında spam filtreleme ve güvenlik doğrulamasında sahte göndericileri engellemek için kritik bir kontrol adımıdır.

---
## 7. Deney 5: E-Posta Mimarisi: SMTP (Push) Diyaloğu Simülasyonu

### 7.1. Deneyin Amacı
Ders notlarınızda belirtilen; e-postanın Alice'ten Bob'un sunucusuna kadar **SMTP (Port 25)** ile nasıl *Push* edildiğini, metin tabanlı komut-yanıt (`HELO`, `MAIL FROM`, `RCPT TO`, `DATA`, `QUIT`) protokol diyaloğunu simüle etmektir.

In [ ]:
# ==============================================================================
# DENEY 5: SMTP E-Posta Gönderim (Push) Protokol Simülatörü
# ==============================================================================
def simule_et_smtp_oturumu(gonderen: str, alici: str, mesaj_icerigi: str):
    print("=" * 65)
    print("SMTP (PORT 25) METİN TABANLI 'PUSH' PROTOKOL DİYALOĞU")
    print("=" * 65)
    
    adimlar = [
        ("Sunucu", "220 mail.bob.com ESMTP Postfix Hazir"),
        ("İstemci (Alice)", f"HELO mail.alice.com"),
        ("Sunucu", "250 Hello mail.alice.com, memnun oldum"),
        ("İstemci (Alice)", f"MAIL FROM: <{gonderen}>"),
        ("Sunucu", "250 2.1.0 Gonderen adresi kabul edildi"),
        ("İstemci (Alice)", f"RCPT TO: <{alici}>"),
        ("Sunucu", "250 2.1.5 Alici adresi kabul edildi"),
        ("İstemci (Alice)", "DATA"),
        ("Sunucu", "354 E-posta govdesini girin; tek satirda '.' ile bitirin"),
        ("İstemci (Alice)", f"Subject: Ag Laboratuvari Notlari\r\n\r\n{mesaj_icerigi}\r\n."),
        ("Sunucu", "250 2.0.0 Mesaj kuyruga alindi (Queued)"),
        ("İstemci (Alice)", "QUIT"),
        ("Sunucu", "221 2.0.0 mail.bob.com Baglanti kapatildi")
    ]
    
    for taraf, komut in adimlar:
        prefix = "-->" if "İstemci" in taraf else "<--"
        print(f"{prefix} [{taraf}]: {komut}")
    print("=" * 65)
    print("[NOT] E-posta Bob'un sunucusuna iletildi (Push). Bob okumak için POP3/IMAP ile 'Pull' yapmalıdır.")

simule_et_smtp_oturumu("alice@cbu.edu.tr", "bob@gmail.com", "Hocam merhaba, lab notlari harika!")

---
### 7.2. Deney 5'in Mühendislik Yorumu
1. **Tek Nokta (`.`) Kuralı:** SMTP protokolünde mesaj gövdesinin bittiğini sunucuya bildiren standart işaret, tek başına yeni bir satıra yazılan nokta (`\r\n.\r\n`) karakteridir.
2. **Push Mimarisi Sınırı:** SMTP sunucuya posta fırlatır ancak alıcının istemcisine zorla mesaj iletemez. Bu yüzden internet e-posta mimarisi iki aşamalıdır: Gönderimde SMTP (Push), alımda ise kullanıcının posta kutusunu sorguladığı POP3 veya IMAP (Pull) çalışır.

---
## 8. Laboratuvar Uygulaması: İki Bağlantılı FTP Protokolü Mimarisi Simülasyonu

### 8.1. Uygulamanın Amacı
Ders notlarında belirtilen; HTTP'den farklı olarak **Out-of-band (Bant Dışı)** çalışan FTP protokolünün çift bağlantı mekanizmasını doğrulamaktır:
* **Port 21 (Control TCP):** Kullanıcı adı, parola doğrulaması ve dizin listeleme komutları için açık kalan kontrol bağlantısı.
* **Port 20 (Data TCP):** Yalnızca dosya aktarımı başladığında açılan ve aktarım bitince anında kapatılan veri bağlantısı.

### 8.2. Yönerge
1. `MockFTPServer` isimli bir Python sınıfı tasarlayın.
2. İstemci önce sunucu ile **Port 21** üzerinden TCP kontrol oturumu açsın (`USER`, `PASS` komutları ile kimlik doğrulasın).
3. Dosya çekme (`RETR dosya.zip`) komutu geldiğinde **Port 20** üzerinde geçici bir veri bağlantısı kurulsun, baytlar transfer edilsin ve veri bağlantısı hemen kapatılsın.
4. Kontrol bağlantısının (Port 21) dosya transferi bittikten sonra bile hala açık kaldığını doğrulayan çıktıları formatlı olarak konsola basın.

In [ ]:
# ==============================================================================
# UYGULAMA: FTP Çift Port (Port 21 Kontrol vs Port 20 Veri) Simülasyonu
# ==============================================================================
class MockFTPServer:
    def __init__(self):
        self.control_port = 21
        self.data_port = 20
        self.authenticated = False
        self.files = {"ders_notu.pdf": b"%PDF-1.4 ... [AKADEMIK AG DERS NOTU ICERIGI] ..."}

    def handle_control_command(self, cmd: str):
        parts = cmd.split()
        action = parts[0]
        if action == "USER":
            return "331 Parola gerekli."
        elif action == "PASS":
            self.authenticated = True
            return "230 Kullanici girisi basarili. Hosgeldiniz."
        elif action == "RETR":
            if not self.authenticated:
                return "530 Once giris yapmalisiniz."
            filename = parts[1]
            if filename in self.files:
                return f"150 Veri baglantisi (Port {self.data_port}) aciliyor. {filename} aktarilacak."
            return "550 Dosya bulunamadi."
        elif action == "QUIT":
            return "221 Kontrol baglantisi kapatiliyor. Hoscakalin."
        return "500 Bilinmeyen komut."

    def transfer_data(self, filename: str):
        print(f"  [DATA TCP - Port {self.data_port}] GEÇİCİ VERİ TÜNELİ AÇILDI.")
        payload = self.files[filename]
        print(f"  [DATA TCP - Port {self.data_port}] {len(payload)} bayt veri aktariliyor...")
        print(f"  [DATA TCP - Port {self.data_port}] AKTARIM TAMAMLANDI. VERİ KANALI KAPATILDI (FIN).")
        return payload

ftp_sunucu = MockFTPServer()

print("=" * 65)
print("FTP PROTOKOLÜ ÇİFT BAĞLANTI (OUT-OF-BAND) SİMÜLASYONU")
print("=" * 65)

print(f"[1] KONTROL BAĞLANTISI (Port {ftp_sunucu.control_port}) AÇILDI.")
print(" İstemci -> USER akin_hoca")
print(f" Sunucu  <- {ftp_sunucu.handle_control_command('USER akin_hoca')}")
print(" İstemci -> PASS gizli_parola")
print(f" Sunucu  <- {ftp_sunucu.handle_control_command('PASS gizli_parola')}")

print(f"\n[2] DOSYA AKTARIM AŞAMASI:")
print(" İstemci -> RETR ders_notu.pdf")
print(f" Sunucu  <- {ftp_sunucu.handle_control_command('RETR ders_notu.pdf')}")
dosya_icerigi = ftp_sunucu.transfer_data("ders_notu.pdf")
print(" Sunucu  <- 226 Dosya basariyla teslim edildi.")

print(f"\n[3] KONTROL KANALI DURUMU (Port {ftp_sunucu.control_port}):")
print(" İstemci -> QUIT")
print(f" Sunucu  <- {ftp_sunucu.handle_control_command('QUIT')}")
print(" KONTROL BAĞLANTISI KAPATILDI.")
print("=" * 65)

---
### 8.3. Konsol Çıktısı ve Mühendislik Yorumu

```text
=================================================================
FTP PROTOKOLÜ ÇİFT BAĞLANTI (OUT-OF-BAND) SİMÜLASYONU
=================================================================
[1] KONTROL BAĞLANTISI (Port 21) AÇILDI.
 İstemci -> USER akin_hoca
 Sunucu  <- 331 Parola gerekli.
 İstemci -> PASS gizli_parola
 Sunucu  <- 230 Kullanici girisi basarili. Hosgeldiniz.

[2] DOSYA AKTARIM AŞAMASI:
 İstemci -> RETR ders_notu.pdf
 Sunucu  <- 150 Veri baglantisi (Port 20) aciliyor. ders_notu.pdf aktarilacak.
  [DATA TCP - Port 20] GEÇİCİ VERİ TÜNELİ AÇILDI.
  [DATA TCP - Port 20] 52 bayt veri aktariliyor...
  [DATA TCP - Port 20] AKTARIM TAMAMLANDI. VERİ KANALI KAPATILDI (FIN).
 Sunucu  <- 226 Dosya basariyla teslim edildi.

[3] KONTROL KANALI DURUMU (Port 21):
 İstemci -> QUIT
 Sunucu  <- 221 Kontrol baglantisi kapatiliyor. Hoscakalin.
 KONTROL BAĞLANTISI KAPATILDI.
=================================================================
```

#### Mühendislik Yorumu:
1. **In-band vs Out-of-band:** HTTP protokolü **In-band (Bant İçi)** çalışır; başlıklar ve HTML gövdesi aynı TCP bağlantısından akar. FTP ise **Out-of-band (Bant Dışı)** çalışır; komutlar Port 21'den, dosya yükü Port 20'den akar.
2. **Yazılım Mühendisliği Açısından Sonuç:** FTP'de büyük bir dosya inerken indirmeyi iptal etmek isterseniz (ABORT), Port 21 kontrol hattı boşta olduğu için komut anında sunucuya ulaşır. Ancak iki farklı port kullanmak günümüz güvenlik duvarları (Firewall) ve NAT yönlendirmelerinde zorluk çıkardığından modern web mimarilerinde dosya transferi için HTTP/HTTPS tercih edilmektedir.